In [0]:
!pip install xgboost

In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd

In [0]:
import mlflow
from mlflow.models import infer_signature
from sklearn.metrics import mean_squared_error



# Route tracking to Databricks and model registry to Unity Catalog
mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")

# Set the MLflow experiment for tracking runs
mlflow.set_experiment(experiment_name="/Shared/housing")
mlflow.set_experiment_tags({"repository_name": "price-prediction"})
print("MLflow experiment configured.")

In [0]:
experiments = mlflow.search_experiments(
    filter_string="tags.repository_name='price-prediction'"
)
print(experiments)

In [0]:
with open("mlflow_experiment.json", "w") as json_file:
    json.dump(experiments[0].__dict__, json_file, indent=4)

In [0]:
housing_df = pd.read_csv("./data/housing.csv")

display(housing_df)

In [0]:
# Use .info() to show features/columns in the dataset, along with data types and counts
housing_df.info()

# Use visualization to understand the relationship of the target variable with other features

Histograms

In [0]:
# Plot the distribution of the target variable (median_house_value) using a histogram

import matplotlib.pyplot as plt

plt.hist(housing_df["median_house_value"],bins=80)
plt.xlabel("House Values")
plt.ylabel("House Count")

# Histogram Interpreration
# Histogram shows outliners on the right.
# Most of the houses are between 100.000 to 200.000.


In [0]:
# Understand data distribution

housing_df.hist(bins=50, figsize=(20,15))

Correlation means: when one thing changes, another thing tends to change along with it.
Examples:

Positive: More study hours → higher grades
Negative: More absences → lower grades
No correlation: Shoe size and intelligence (no relationship)

It ranges from -1 to +1, where 1 = perfect match, 0 = no relationship, -1 = perfect opposite.

In [0]:
# Plot a graphical correlation matrix for each pair of columns in the dataframe
corr = housing_df.corr()
print(corr)

In [0]:
# https://www.linkedin.com/learning/artificial-intelligence-foundations-machine-learning-22345868/visualizing-and-understanding-data?autoSkip=true&resume=false&u=0

import seaborn as sns 

# make the heatmap larger in size
plt.figure(figsize = (8,8))

sns.heatmap(corr, annot=True)
plt.show()

In [0]:
# Feature engineering for improving the model prediction



Prepare and process dataa

In [0]:
# Verify which features havemissing values
housing_df.isnull().sum()

In [0]:
# Calculate the % of missing data
housing_df["total_bedrooms"].isnull().sum()/housing_df.shape[0] * 100

Use impulation to handle data

In [0]:
from sklearn.impute import KNNImputer

# create a temporary copy of the dataset
housing_df_temp = housing_df.copy()

# retrieve columns with numerical data; will exclude the ocean_proximity column since the datatype is object; other columns are float64
columns_list = [col for col in housing_df_temp.columns if housing_df_temp[col].dtype != 'object']

# extract columns that contain at least one missing value
new_column_list = [col for col in housing_df_temp.loc[:, housing_df_temp.isnull().any()]]

# update temp dataframe with numeric columns that have empty values
housing_df_temp = housing_df_temp[new_column_list]



impute missing data using machine learning

In [0]:
# initialize KNNImputer to impute missing data using machine learning
knn = KNNImputer(n_neighbors = 3)

# fit function trains the model 
knn.fit(housing_df_temp)

# transform the data using the model
# applies the transformation model (ie knn) to data
array_Values = knn.transform(housing_df_temp)

# convert the array values to a dataframe with the appropriate column names
housing_df_temp = pd.DataFrame(array_Values, columns = new_column_list)
display(housing_df_temp)


In [0]:

# confirm there are no columns with missing values
housing_df_temp.isnull().sum()

In [0]:

# overlay the imputed column over the old column with missing values

# loop through the list of columns and overlay each one
for column_name in new_column_list:
    housing_df[column_name] = housing_df_temp.replace(housing_df[column_name],housing_df[column_name])

# confirm columns no longer contain null data
housing_df.isnull().sum()

In [0]:

# Additionally we noted that several features (total_rooms,total_bedrooms,population,households) have very high correlation to one another, 
# so it's interesting to find out if a removal of a few of them would have any affect on the model performance

#  a new feature that is a ratio of the total rooms to households
housing_df['rooms_per_household'] = housing_df['total_rooms']/housing_df['households']

# a new feature that is a ratio of the total bedrooms to the total rooms 
housing_df['bedrooms_per_room'] = housing_df['total_bedrooms']/housing_df['total_rooms']

# a new feature that is a ratio of the population to the households 
housing_df['population_per_household']= housing_df['population']/housing_df['households']

# let's combine the latitude and longitude into 1
housing_df['coords'] = housing_df['longitude']/housing_df['latitude']

housing_df.info()

In [0]:
# remove total_rooms, households, total bedrooms, popluation, longitude, latitude
housing_df = housing_df.drop('total_rooms', axis=1)
housing_df = housing_df.drop('households', axis=1)
housing_df = housing_df.drop('total_bedrooms', axis=1)
housing_df = housing_df.drop('population', axis=1)
housing_df = housing_df.drop('longitude', axis=1)
housing_df = housing_df.drop('latitude', axis=1)

housing_df.info()

In [0]:
corr = housing_df.corr(numeric_only=True) 

#make the heatmap larger in size
plt.figure(figsize = (7,7))

sns.heatmap(corr, annot=True)
plt.show()

In [0]:

# let's see the unique categories for OCEAN_PROXIMITY
housing_df.ocean_proximity.unique()

In [0]:
# let's count
housing_df["ocean_proximity"].value_counts()

In [0]:

# Let's see how the Panda's get_dummies() function works
print(pd.get_dummies(housing_df['ocean_proximity']))

In [0]:

# let's replace the OCEAN_PROXIMITY column using get_dummies()
housing_df_encoded = pd.get_dummies(data=housing_df, columns=['ocean_proximity'])

# print the first few observations; notice the old OCEAN_PROXIMITY column is gone
housing_df_encoded.head()

In [0]:
import sklearn
from sklearn.model_selection import train_test_split

# remove spaces from column names and convert all to lowercase and remove special characters as it could cause issues in the future
housing_df_encoded.columns = [c.lower().replace(' ', '_').replace('<', '_') for c in housing_df_encoded.columns]

# Split target variable and feature variables
X = housing_df_encoded[['housing_median_age', 'median_income','bedrooms_per_room','population_per_household','coords','ocean_proximity__1h_ocean',
                        'ocean_proximity_inland','ocean_proximity_island','ocean_proximity_near_bay','ocean_proximity_near_ocean']]
y = housing_df_encoded['median_house_value']

print(X)

In [0]:

# Splitting the data into training and testing sets in numpy arrays
# We train the model with 70% of the samples and test with the remaining 30%
# X -> array with the inputs; y -> array of the outputs
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, shuffle=True, test_size=0.3)

# Confirm how the data was split
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

In [0]:
# Pipelines

# Import pipeline code
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

pipeline_lr  = Pipeline([('lr_classifier',  LinearRegression())])
pipeline_rf  = Pipeline([('rf_classifier',  RandomForestRegressor(n_estimators=10, random_state=10))])
pipeline_xgb = Pipeline([('xgb_classifier', XGBRegressor())])

pipelines = [pipeline_lr, pipeline_rf, pipeline_xgb]
pipe_dict = {0: 'Linear Regression', 1: 'Random Forest Regressor', 2: 'XGBRegressor'}

for i, pipe in enumerate(pipelines):
    model_name = pipe_dict[i]
    with mlflow.start_run(run_name=model_name):
        pipe.fit(X_train, y_train)
        pred_test = pipe.predict(X_test)
        r2   = pipe.score(X_test, y_test)
        rmse = mean_squared_error(y_test, pred_test) ** 0.5

        # Log parameters
        mlflow.log_param("model_type",   model_name)
        mlflow.log_param("test_size",    0.3)
        mlflow.log_param("random_state", 42)

        # Log metrics
        mlflow.log_metric("r2_score", r2)
        mlflow.log_metric("rmse",     rmse)

        # Log model with inferred signature
        signature = infer_signature(X_train, pipe.predict(X_train))
        mlflow.sklearn.log_model(
            pipe,
            name="model",
            signature=signature,
            input_example=X_train[:5],
        )

        if i == 1:
            rf_model = pipe

        print(f"{model_name}  |  R²: {r2:.4f}  |  RMSE: {rmse:.2f}")
        print(pd.DataFrame({'Actual': y_test, 'Predicted': pred_test}))

In [0]:

# Determine feature importance - random forest algorithm is that it gives you the ‘feature importance’ for all the variables in the data
# plot the 5 most important features 
plt.figure(figsize=(10,5))
feat_importances = pd.Series(rf_model['rf_classifier'].feature_importances_, index = X_train.columns)
feat_importances.nlargest(5).plot(kind='barh');

In [0]:
# Training data with 5 most important features
train_x_if = X_train[['housing_median_age', 'coords', 'ocean_proximity_inland', 'population_per_household', 'median_income']]
test_x_if  = X_test[['housing_median_age',  'coords', 'ocean_proximity_inland', 'population_per_household', 'median_income']]

with mlflow.start_run(run_name="Random Forest - Top 5 Features"):
    # Create and train the model
    rf_model_if = RandomForestRegressor(n_estimators=10, random_state=10)
    rf_model_if.fit(train_x_if, y_train)

    # Predict the target on the test data
    predict_test_with_if = rf_model_if.predict(test_x_if)
    rmse = mean_squared_error(y_test, predict_test_with_if) ** 0.5
    r2   = rf_model_if.score(test_x_if, y_test)

    # Log parameters
    mlflow.log_param("model_type",    "RandomForestRegressor")
    mlflow.log_param("n_estimators",  10)
    mlflow.log_param("random_state",  10)
    mlflow.log_param("feature_set",   "top_5_important")
    mlflow.log_param("features",      "housing_median_age,coords,ocean_proximity_inland,population_per_household,median_income")

    # Log metrics
    mlflow.log_metric("rmse",     rmse)
    mlflow.log_metric("r2_score", r2)

    # Log model with inferred signature
    signature = infer_signature(train_x_if, rf_model_if.predict(train_x_if))
    mlflow.sklearn.log_model(
        rf_model_if,
        name="model",
        signature=signature,
        input_example=train_x_if[:5],
    )

    print(f"Run logged: Random Forest - Top 5 Features  |  R²: {r2:.4f}  |  RMSE: {rmse:.2f}")

In [0]:

# Root Mean Squared Error on the train and test data
print('RMSE on test data: ',  mean_squared_error(y_test, predict_test_with_if)**(0.5))